# Quantization-Aware Training (QAT)

Normally, Post-Training Quantization (PTQ) is applied after training finishes:

$$
\text{Base Model} \xrightarrow{\text{SFT}} \text{Fine-Tuned FP16} \xrightarrow{\text{PTQ Compression}} \text{4-bit Artifact}
$$

During PTQ, rounding errors and outlier clipping cause sudden quality drops on narrow domain tasks.

## 1. The QAT mechanism

Quantization-Aware Training (QAT) simulates quantization noise during the forward pass of fine-tuning so the LoRA adapters adapt to the low-bit representation before export.

```text
QAT Forward Pass (Simulated Quantization)
[ Weight W (FP32) ] ──► [ Fake Quantize & Clamp to 4-bit Grid ] ──► [ Simulated W_quant ]
                                                                             │
                                                                             ▼
                                                                 [ Forward Pass MatMul ]
                                                                             │
                                                                             ▼
                                                                  [ Calculate Task Loss ]
```

## Mathematical formulation (Straight-Through Estimator - STE)

During the forward pass, weights are mapped to discrete 4-bit values:

$$
W_{\text{quant}} = \text{clamp}\left( \text{round}\left(\frac{W}{S}\right), -8, 7 \right) \times S
$$

Where $S$ is the learned quantization scale.

## The backward pass challenge

The derivative of the rounding function $\text{round}(x)$ is zero almost everywhere:

$$
\frac{d}{dx} \text{round}(x) = 0
$$

which would block backpropagation gradients entirely.

QAT uses the Straight-Through Estimator (STE):

$$
\frac{\partial W_{\text{quant}}}{\partial W} =
\begin{cases}
1 & \text{if } \lvert W / S \rvert \le \text{Threshold} \\
0 & \text{otherwise}
\end{cases}
$$

The gradient passes straight through the quantization operator as if it were an identity function. The trainable LoRA parameters actively learn to compensate for the rounding errors of the 4-bit base model.